# 08 · Robustness and QC Sensitivity Checks

**Goal**: Assess the robustness of clustering results and their sensitivity to QC thresholds and alternative clustering resolutions.

**Input**: results/adata/07_annotated.h5ad
**Output**: results/adata/08_robustness.h5ad, robustness summary tables, and figures.

**Note**: This is a diagnostic analysis, not biological validation. No QC filtering is applied to the original dataset, and the existing Leiden clusters and provisional annotations are preserved. Results remain exploratory.

## 1. Imports and paths

In [ ]:
from pathlib import Path
from importlib.metadata import version
import sys
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from scipy import sparse, stats
from sklearn.metrics import adjusted_rand_score

def setup_robustness_paths():
    ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "results/adata/07_annotated.h5ad").exists()), None)
    if ROOT is None:
        raise FileNotFoundError("Run 07_biological_annotation first; missing results/adata/07_annotated.h5ad")
    INPUT = ROOT / "results/adata/07_annotated.h5ad"
    OUTPUT = ROOT / "results/adata/08_robustness.h5ad"
    TAB = ROOT / "results/tables"
    FIG = ROOT / "results/figures"
    for p in (OUTPUT.parent, TAB, FIG): p.mkdir(parents=True, exist_ok=True)
    SEED = 42
    rng = np.random.default_rng(SEED)
    print("Python:", sys.executable)
    print("Scanpy:", version("scanpy"), "Project:", ROOT)
    return ROOT, INPUT, OUTPUT, TAB, FIG, SEED, rng

ROOT, INPUT, OUTPUT, TAB, FIG, SEED, rng = setup_robustness_paths()

## 2. Load and verify data integrity

Load the annotated AnnData from Step 07 and verify its dimensions, spatial coordinates, expression and spatial neighbor graphs, raw count layer, QC metrics, and provisional annotations.

Preserve the original Leiden cluster labels for comparison during robustness checks. Report the dataset dimensions, number of clusters, and spatial graph connections.

In [ ]:
def load_and_validate_annotations(input_path):
    adata = sc.read_h5ad(input_path)
    assert adata.shape == (14400, 18085), f'Unexpected dataset shape: {adata.shape}'
    assert 'leiden' in adata.obs and 'spatial' in adata.obsm
    assert 'counts' in adata.layers and sparse.issparse(adata.layers['counts'])
    assert all((k in adata.obsp for k in ('connectivities', 'distances', 'spatial_connectivities', 'spatial_distances')))
    assert all((k in adata.obs for k in ('total_counts', 'n_genes_by_counts')))
    assert 'provisional_domain_program' in adata.obs
    original_leiden = adata.obs['leiden'].copy()
    clusters = list(adata.obs['leiden'].cat.categories)
    print('Data:', adata.shape, 'Clusters:', len(clusters), 'Spatial edges:', adata.obsp['spatial_connectivities'].nnz)
    return adata, original_leiden, clusters

adata, original_leiden, clusters = load_and_validate_annotations(INPUT)

## 3. Cluster size imbalance and QC summaries

Summarize cluster sizes and QC metrics (total counts, detected genes, and mitochondrial percentage when available). Export the summary table and cluster size plot.

These results are descriptive and do not establish technical confounding.

In [ ]:
def summarize_cluster_qc(adata, tab_dir, fig_dir):
    qc_cols = ['total_counts', 'n_genes_by_counts']
    if 'pct_counts_mt' in adata.obs and pd.to_numeric(adata.obs['pct_counts_mt'], errors='coerce').notna().any():
        qc_cols.append('pct_counts_mt')
    obs = adata.obs.copy()
    obs['leiden'] = obs['leiden'].astype(str)
    for col in qc_cols:
        obs[col] = pd.to_numeric(obs[col], errors='coerce')
    rows = []
    for cl, frame in obs.groupby('leiden', sort=False):
        row = {'leiden': cl, 'n_bins': len(frame), 'fraction': len(frame) / len(obs)}
        for col in qc_cols:
            v = frame[col].dropna()
            row[f'{col}_median'] = float(v.median()) if len(v) else np.nan
            row[f'{col}_q25'] = float(v.quantile(0.25)) if len(v) else np.nan
            row[f'{col}_q75'] = float(v.quantile(0.75)) if len(v) else np.nan
        rows.append(row)
    cluster_qc = pd.DataFrame(rows).sort_values('n_bins', ascending=False)
    cluster_qc.to_csv(tab_dir / '08_cluster_qc_summary.csv', index=False)
    print(cluster_qc.head(10).to_string(index=False))
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.bar(cluster_qc['leiden'], cluster_qc['n_bins'])
    ax.set(xlabel='Leiden cluster', ylabel='Number of 56 µm bins', title='Cluster size imbalance (original resolution 0.5)')
    ax.tick_params(axis='x', rotation=90)
    fig.tight_layout()
    fig.savefig(fig_dir / '08_cluster_sizes.png', dpi=170)
    plt.show()
    return qc_cols, obs, cluster_qc

qc_cols, obs, cluster_qc = summarize_cluster_qc(adata, TAB, FIG)

## 4. QC association with cluster assignments

Use Kruskal–Wallis tests to assess differences in QC metrics across Leiden clusters. Report approximate effect sizes and visualize QC distributions using boxplots.

Export the results to 08_qc_cluster_associations.csv and 08_qc_by_cluster.png.

These analyses are exploratory; spatial dependence limits the interpretation of p-values.

In [ ]:
def analyze_qc_cluster_associations(obs, qc_cols, clusters, tab_dir, fig_dir):
    association = []
    for col in qc_cols:
        groups = [g[col].dropna().to_numpy(dtype=float) for _, g in obs.groupby('leiden', sort=False)]
        groups = [v for v in groups if len(v)]
        if len(groups) >= 2 and len(np.unique(np.concatenate(groups))) > 1:
            h, p = stats.kruskal(*groups)
            n = sum(map(len, groups))
            k = len(groups)
            eps2 = float(np.clip((h - k + 1) / max(1, n - k), 0, 1))
        else:
            h, p, eps2 = (np.nan, np.nan, np.nan)
        association.append({'qc_metric': col, 'kruskal_H': h, 'nominal_p': p, 'epsilon_squared_approx': eps2, 'note': 'exploratory; bins spatially dependent'})
    qc_tests = pd.DataFrame(association)
    qc_tests.to_csv(tab_dir / '08_qc_cluster_associations.csv', index=False)
    print(qc_tests.to_string(index=False))
    fig, axes = plt.subplots(1, len(qc_cols), figsize=(max(8, 5 * len(qc_cols)), 5), squeeze=False)
    for ax, col in zip(axes[0], qc_cols):
        data = [obs.loc[obs['leiden'] == cl, col].dropna().to_numpy() for cl in clusters]
        ax.boxplot(data, showfliers=False, tick_labels=clusters)
        ax.set(title=col, xlabel='Leiden', ylabel=col)
        ax.tick_params(axis='x', rotation=90, labelsize=7)
    fig.suptitle('QC distributions by original cluster (outliers hidden)')
    fig.tight_layout()
    fig.savefig(fig_dir / '08_qc_by_cluster.png', dpi=170)
    plt.show()
    return qc_tests

qc_tests = analyze_qc_cluster_associations(obs, qc_cols, clusters, TAB, FIG)

## 5. Resolution sensitivity of Leiden clustering

Recompute Leiden clusters at resolutions 0.3, 0.5, 0.8, and 1.0 using the same expression-neighbor graph and random seed.

Compare cluster assignments using the Adjusted Rand Index (ARI), and export summary tables and a heatmap.

This evaluates sensitivity to clustering resolution, not overall clustering stability. Original Leiden labels are preserved.

In [ ]:
def analyze_resolution_sensitivity(adata, original_leiden, tab_dir, fig_dir, seed):
    RESOLUTIONS = [0.3, 0.5, 0.8, 1.0]
    labels = {'original_0.5': original_leiden.astype(str).to_numpy()}
    resolution_rows = []
    for res in RESOLUTIONS:
        key = 'leiden_r' + str(res).replace('.', 'p')
        sc.tl.leiden(adata, resolution=res, key_added=key, random_state=seed, flavor='igraph', directed=False)
        values = adata.obs[key].astype(str).to_numpy()
        labels[key] = values
        resolution_rows.append({'resolution': res, 'key': key, 'n_clusters': int(adata.obs[key].nunique()), 'ARI_vs_original': float(adjusted_rand_score(labels['original_0.5'], values))})
    resolution_df = pd.DataFrame(resolution_rows)
    resolution_df.to_csv(tab_dir / '08_resolution_sensitivity.csv', index=False)
    print(resolution_df.to_string(index=False))
    keys = list(labels)
    ari = pd.DataFrame([[adjusted_rand_score(labels[a], labels[b]) for b in keys] for a in keys], index=keys, columns=keys)
    ari.to_csv(tab_dir / '08_resolution_pairwise_ari.csv')
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(ari, vmin=0, vmax=1, cmap='viridis')
    ax.set_xticks(range(len(keys)), keys, rotation=60, ha='right')
    ax.set_yticks(range(len(keys)), keys)
    for i in range(len(keys)):
        for j in range(len(keys)):
            ax.text(j, i, f'{ari.iloc[i, j]:.2f}', ha='center', va='center', color='white' if ari.iloc[i, j] < 0.5 else 'black')
    fig.colorbar(im, ax=ax, label='Adjusted Rand Index')
    ax.set_title('Resolution sensitivity (shared expression graph)')
    fig.tight_layout()
    fig.savefig(fig_dir / '08_resolution_ari.png', dpi=170)
    plt.show()
    return RESOLUTIONS, resolution_df, ari

RESOLUTIONS, resolution_df, ari = analyze_resolution_sensitivity(adata, original_leiden, TAB, FIG, SEED)

## 6. Spatial continuity: within-cluster edge fraction

Calculate the fraction of spatial neighbor connections shared by bins with the same original Leiden label.

Compare the observed fraction with 100 random label permutations and export a summary table and histogram.

This is a descriptive assessment of spatial continuity, not a formal statistical significance test.

In [ ]:
def analyze_spatial_continuity(adata, original_leiden, tab_dir, fig_dir, rng):
    W = adata.obsp['spatial_connectivities'].tocsr()
    W = W.maximum(W.T).tocsr()
    W.setdiag(0)
    W.eliminate_zeros()
    r, col = sparse.triu(W, k=1).nonzero()
    assert len(r) > 0, 'No spatial edges found'
    code = original_leiden.cat.codes.to_numpy()
    observed = float(np.mean(code[r] == code[col]))
    N_PERMS = 100
    null = np.empty(N_PERMS, dtype=float)
    for i in range(N_PERMS):
        shuffled = rng.permutation(code)
        null[i] = np.mean(shuffled[r] == shuffled[col])
    spatial_summary = pd.DataFrame([{'n_undirected_edges': len(r), 'within_cluster_fraction': observed, 'shuffled_mean': float(null.mean()), 'shuffled_sd': float(null.std(ddof=1)), 'n_shuffles': N_PERMS, 'interpretation': 'descriptive; shuffled labels ignore spatial dependence'}])
    spatial_summary.to_csv(tab_dir / '08_spatial_continuity.csv', index=False)
    print(spatial_summary.to_string(index=False))
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(null, bins=20, alpha=0.7, label='Shuffled labels')
    ax.axvline(observed, color='crimson', lw=2, label='Observed')
    ax.set(xlabel='Fraction of same-cluster spatial edges', ylabel='Count', title='Spatial continuity: descriptive shuffled-label reference')
    ax.legend()
    fig.tight_layout()
    fig.savefig(fig_dir / '08_spatial_continuity.png', dpi=170)
    plt.show()
    return N_PERMS, spatial_summary

N_PERMS, spatial_summary = analyze_spatial_continuity(adata, original_leiden, TAB, FIG, rng)

## 7. Exploratory QC threshold sensitivity (without filtering)

Estimate bin retention across Leiden clusters under illustrative QC thresholds. Export a summary table and plot.

No filtering or reclustering is performed.

In [ ]:
def evaluate_hypothetical_qc_retention(obs, qc_cols, clusters, tab_dir, fig_dir):
    MIN_COUNTS = 100
    MIN_GENES = 50
    MAX_PCT_MT = 25.0
    mask = (obs['total_counts'].to_numpy() >= MIN_COUNTS) & (obs['n_genes_by_counts'].to_numpy() >= MIN_GENES)
    mt_used = 'pct_counts_mt' in qc_cols
    if mt_used:
        mt = obs['pct_counts_mt'].to_numpy(dtype=float)
        mask &= np.isfinite(mt) & (mt <= MAX_PCT_MT)
    retention = []
    for cl in clusters:
        idx = obs['leiden'].to_numpy() == cl
        retention.append({'leiden': cl, 'n_bins': int(idx.sum()), 'would_retain': int(mask[idx].sum()), 'would_retain_fraction': float(mask[idx].mean())})
    retention_df = pd.DataFrame(retention)
    retention_df.to_csv(tab_dir / '08_hypothetical_qc_retention.csv', index=False)
    print('Illustrative thresholds:', {'min_counts': MIN_COUNTS, 'min_genes': MIN_GENES, 'max_pct_mt': MAX_PCT_MT if mt_used else 'not available'})
    print('Would retain:', int(mask.sum()), '/', len(mask), 'bins. NO FILTERING APPLIED.')
    print(retention_df.sort_values('would_retain_fraction').head(10).to_string(index=False))
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.bar(retention_df['leiden'], retention_df['would_retain_fraction'])
    ax.set(xlabel='Leiden cluster', ylabel='Hypothetical retention fraction', ylim=(0, 1.02), title='Illustrative QC thresholds — not applied')
    ax.tick_params(axis='x', rotation=90)
    fig.tight_layout()
    fig.savefig(fig_dir / '08_hypothetical_qc_retention.png', dpi=170)
    plt.show()
    return retention_df

retention_df = evaluate_hypothetical_qc_retention(obs, qc_cols, clusters, TAB, FIG)

## 8. Save and reload validation

Save the robustness results and processing settings to 08_robustness.h5ad, then reload the file to verify data integrity.

Confirm that the original Leiden labels, spatial graphs, raw counts, and provisional annotations are preserved.

These checks are diagnostic and do not validate biological cell identities.

In [ ]:
def save_and_verify_robustness(adata, original_leiden, resolutions, seed, n_perms, output_path, tab_dir, fig_dir, root):
    assert adata.obs['leiden'].equals(original_leiden), 'Original cluster labels changed'
    adata.uns['step08_settings'] = {'diagnostic_only': True, 'qc_filters_applied': False, 'resolution_values': resolutions, 'seed': seed, 'spatial_shuffles': n_perms, 'spatial_null_calibrated': False, 'image_registration_verified': False}
    adata.write_h5ad(output_path, compression='gzip')
    check = sc.read_h5ad(output_path)
    assert check.shape == adata.shape
    assert check.obs['leiden'].equals(original_leiden)
    assert np.array_equal(check.obsm['spatial'], adata.obsm['spatial'])
    assert all((k in check.obsp for k in ('connectivities', 'distances', 'spatial_connectivities', 'spatial_distances')))
    assert 'counts' in check.layers and sparse.issparse(check.layers['counts'])
    assert 'provisional_domain_program' in check.obs
    assert 'step08_settings' in check.uns
    assert (tab_dir / '08_resolution_sensitivity.csv').exists()
    assert (fig_dir / '08_resolution_ari.png').exists()
    print('PASS — 08_robustness_checks completed')
    print('AnnData:', check.shape)
    print('Original Leiden clusters:', check.obs['leiden'].nunique())
    print('Resolution variants:', len(resolutions))
    print('Output:', output_path.relative_to(root))
    return check

check = save_and_verify_robustness(adata, original_leiden, RESOLUTIONS, SEED, N_PERMS, OUTPUT, TAB, FIG, ROOT)

### Completion · Step 08

Review QC associations, spatial continuity, and Leiden resolution sensitivity before drawing biological conclusions.

QC thresholds were evaluated without applying filters. Original Leiden cluster labels and provisional annotations remain unchanged.

**Output**: results/adata/08_robustness.h5ad

**Workflow complete**: Steps 01–08. All findings remain exploratory and do not constitute validated cell-type identities.